In [1]:
import pandas as pd
from scipy import stats
import sys
from pathlib import Path
sys.path.append('../src')
from src.extract_starting_agent import get_agent_with_opening_statement, get_agent_who_starts

# Analysis for GPT-OSS and Mistral
## GPT-OSS

In [2]:
df_oss = pd.read_csv("../data/evaluation mad/convergence_of_3986_toxic_random_discussions_topic_16_oss.csv", sep =",")

In [20]:
dt_oss = df_oss[df_oss["reason_for_convergence"] != "moderator detected alignment"]
dt_oss = dt_oss.copy()
dt_oss["toxic_agent"] = dt_oss.apply(lambda row: 0 if row["which_agent_is_toxic"] == "pro has toxic behaviour" else 1 , axis=1) # 0 for pro is toxic, 1 for con is toxic

### AVOVA: Toxicity Level has an effect on the rounds till convergence (equal to the paper MAD 1)


In [4]:
groups = [group["rounds_to_convergence"].values for _, group in dt_oss.groupby("toxicity_level")]

f_stat, p_value = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# group means for overview
print(dt_oss.groupby("toxicity_level")["rounds_to_convergence"].mean())

F-statistic: 163.3101
P-value: 0.0000
toxicity_level
heavy       13.992126
mild        11.371105
moderate    13.336585
no           8.001642
Name: rounds_to_convergence, dtype: float64


# Is the Toxic Agent winning?


### One-Sample T-Test: The Toxic Agent (PRO and CON) is winning significantly more than 50% of the time

In [5]:
# when pro is toxic, does pro win?
pro_toxic = (dt_oss[dt_oss["toxic_agent"] == 0]["reason_for_convergence"] == "con has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(pro_toxic, popmean=0.5)
print(f"PRO toxic:")
print(f"  win rate: {pro_toxic.mean():.4f}")
print(f"  p-value: {p_value:.4f}")

# when con is toxic, does con win?
con_toxic = (dt_oss[dt_oss["toxic_agent"] == 1]["reason_for_convergence"] == "pro has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(con_toxic, popmean=0.5)
print(f"CON toxic:")
print(f"  win rate: {con_toxic.mean():.4f}")
print(f"  p-value: {p_value:.4f}")

PRO toxic:
  win rate: 0.6384
  p-value: 0.0000
CON toxic:
  win rate: 0.6137
  p-value: 0.0000


### Two-Sample T-Test: Both PRO and CON toxic agents are winning significantly more than their non-toxic counterparts.
### PRO

In [6]:
# when pro is toxic (toxic_agent == 0), pro wins = con gets convinced
pro_toxic_wins = (dt_oss[dt_oss["toxic_agent"] == 0]["reason_for_convergence"] == "con has been convinced").astype(int)

# when con is toxic (toxic_agent == 1), pro is NOT toxic -> pro wins = con gets convinced
pro_nontoxic_wins = (dt_oss[dt_oss["toxic_agent"] == 1]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(pro_toxic_wins, pro_nontoxic_wins)
print(f"PRO toxic win rate: {pro_toxic_wins.mean():.4f}")
print(f"PRO non-toxic win rate: {pro_nontoxic_wins.mean():.4f}")
print(f"p-value: {p_value:.4f}")

PRO toxic win rate: 0.6384
PRO non-toxic win rate: 0.3863
p-value: 0.0000


### CON

In [7]:
con_toxic_wins = (dt_oss[dt_oss["toxic_agent"] == 1]["reason_for_convergence"] == "pro has been convinced").astype(int)
con_nontoxic_wins = (dt_oss[dt_oss["toxic_agent"] == 0]["reason_for_convergence"] == "pro has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(con_toxic_wins, con_nontoxic_wins)
print(f"CON toxic win rate: {con_toxic_wins.mean():.4f}")
print(f"CON non-toxic win rate: {con_nontoxic_wins.mean():.4f}")
print(f"p-value: {p_value:.4f}")

CON toxic win rate: 0.6137
CON non-toxic win rate: 0.3616
p-value: 0.0000


### AVOVA: Toxicity level has a significant effect on the winning rate of the Toxic Agent.

In [8]:
dt_oss = dt_oss.copy()
dt_oss["pro_wins"] = (dt_oss["reason_for_convergence"] == "con has been convinced").astype(int)
dt_oss["con_wins"] = (dt_oss["reason_for_convergence"] == "pro has been convinced").astype(int)

In [9]:
groups_pro = [group["pro_wins"].values for _, group in dt_oss.groupby("toxicity_level")]
f_stat, p_value = stats.f_oneway(*groups_pro)
print(f"PRO wins by toxicity:")
print(f"  F-statistic: {f_stat:.4f}, P-value: {p_value:.4f}")
print(dt_oss.groupby("toxicity_level")["pro_wins"].mean())

PRO wins by toxicity:
  F-statistic: 11.1989, P-value: 0.0000
toxicity_level
heavy       0.436220
mild        0.569405
moderate    0.452033
no          0.538588
Name: pro_wins, dtype: float64


In [10]:
groups_con = [group["con_wins"].values for _, group in dt_oss.groupby("toxicity_level")]
f_stat, p_value = stats.f_oneway(*groups_con)
print(f"CON wins by toxicity:")
print(f"  F-statistic: {f_stat:.4f}, P-value: {p_value:.4f}")
print(dt_oss.groupby("toxicity_level")["con_wins"].mean())

CON wins by toxicity:
  F-statistic: 11.1989, P-value: 0.0000
toxicity_level
heavy       0.563780
mild        0.430595
moderate    0.547967
no          0.461412
Name: con_wins, dtype: float64


# Is the Agent winning that starts the discussion?


In [11]:
directory = Path('../data/toxic_and_baseline_random_oss')
dt_oss["opening_agent"] = dt_oss["Path"].apply(get_agent_with_opening_statement(directory))
dt_oss["starting_agent"] = dt_oss["Path"].apply(get_agent_who_starts(directory))

In [12]:
dt_oss.groupby("opening_agent")["reason_for_convergence"].value_counts(normalize=True)

opening_agent  reason_for_convergence
con            pro has been convinced    0.502269
               con has been convinced    0.497731
pro            con has been convinced    0.504425
               pro has been convinced    0.495575
Name: proportion, dtype: float64

### T-Test: The Agent who starts the discussion does have a significant advantage in winning the debate.

In [15]:
con_winning = (dt_oss[dt_oss["starting_agent"] == "con"]["reason_for_convergence"] == "pro has been convinced").astype(int)
con_loosing = (dt_oss[dt_oss["starting_agent"] == "con"]["reason_for_convergence"] == "con has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(con_winning, con_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

pro_winning = (dt_oss[dt_oss["starting_agent"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)
pro_loosing = (dt_oss[dt_oss["starting_agent"] == "pro"]["reason_for_convergence"] == "pro has been convinced").astype(int)

t_stat, p_value = stats.ttest_ind(pro_winning, pro_loosing)
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

t-statistic: 17.4485
p-value: 0.0000
t-statistic: 17.5051
p-value: 0.0000


### One-Sample T-Test: Starting Agent is winning significantly more than 50% of the time (Equally for PRO and CON)

In [22]:
# when con starts, do they win more than 50%?
con_outcomes = (dt_oss[dt_oss["starting_agent"] == "con"]["reason_for_convergence"] == "pro has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(con_outcomes, popmean=0.5)
print(f"CON starter:")
print(f"  win rate: {con_outcomes.mean():.4f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

# when pro starts, do they win more than 50%?
pro_outcomes = (dt_oss[dt_oss["starting_agent"] == "pro"]["reason_for_convergence"] == "con has been convinced").astype(int)
t_stat, p_value = stats.ttest_1samp(pro_outcomes, popmean=0.5)
print(f"PRO starter:")
print(f"  win rate: {pro_outcomes.mean():.4f}")
print(f"  t-statistic: {t_stat:.4f}")
print(f"  p-value: {p_value:.4f}")

KeyError: 'starting_agent'

## Mistral

In [17]:
df_mistral = pd.read_csv("../data/evaluation mad/.csv", sep =",")
dt_mistral = df_mistral[df_mistral["reason_for_convergence"] != "moderator detected alignment"]

FileNotFoundError: [Errno 2] No such file or directory: '../data/evaluation mad/.csv'

### AVOVA to check if the toxicity level has an effect on the rounds till convergence (equal to the paper MAD 1)


In [ ]:
groups = [group["rounds_to_convergence"].values for _, group in df_mistral.groupby("toxicity_level")]

f_stat, p_value = stats.f_oneway(*groups)
print(f"F-statistic: {f_stat:.4f}")
print(f"P-value: {p_value:.4f}")

# group means for overview
print(df_mistral.groupby("toxicity_level")["rounds_to_convergence"].mean())

# Is the Toxic Agent winning?